# Metadatos, Catálogo y Linaje de Datos (Mid)

Objetivos:
- Entender tipos de metadatos (técnicos, de negocio, operacionales).
- Conocer catálogos de datos y su rol en gobernanza.
- Comprender linaje de datos (data lineage) para auditoría y trazabilidad.

## 1. ¿Qué son los Metadatos?

**Metadatos = datos sobre los datos.**

### Tipos:
- **Técnicos:** Esquema (columnas, tipos, constraints), tamaño, formato, ubicación.
- **De negocio:** Definiciones, glosario, propietarios, clasificación (PII, confidencial).
- **Operacionales:** Fecha de creación/última actualización, estadísticas (conteos, nulls), SLAs.

Ejemplo:
```
Tabla: clientes
- Metadato técnico: columnas (id INT, nombre VARCHAR(100), email VARCHAR)
- Metadato de negocio: "Registro de clientes activos. Owner: Comercial."
- Metadato operacional: última carga 2025-11-07 06:00, 12,543 filas.
```

## 2. Catálogo de Datos

Repositorio centralizado de metadatos que permite:
- **Descubrimiento:** Buscar datasets por nombre, etiquetas, columna.
- **Documentación:** Descripciones, lineage, ownership.
- **Gobernanza:** Clasificación, políticas de acceso, calidad.

Herramientas populares:
- Open-source: Apache Atlas, Amundsen, DataHub.
- Comerciales: Atlan, Collibra, Alation, Google Data Catalog, AWS Glue Catalog.

## 3. Ejemplo de Metadatos en Python (Simulado)

In [ ]:
import pandas as pd
from datetime import datetime

# Simulación de tabla y metadatos
df = pd.DataFrame({
    'cliente_id': [1,2,3,4],
    'nombre': ['Ana','Luis','Carlos','Marta'],
    'email': ['ana@x.com','luis@y.com',None,'marta@z.com'],
    'saldo': [1500, 2300, 800, 3200]
});

metadatos_tecnicos = {
    'tabla': 'clientes',
    'columnas': list(df.columns),
    'tipos': dict(df.dtypes.astype(str)),
    'filas': len(df),
    'ubicacion': 's3://datalake/silver/clientes.parquet'
}

metadatos_negocio = {
    'descripcion': 'Tabla de clientes activos',
    'owner': 'Equipo Comercial',
    'clasificacion': 'Confidencial (contiene email)',
    'dominio': 'ventas'
}

metadatos_operacionales = {
    'ultima_actualizacion': datetime.utcnow().isoformat(),
    'nulls': df.isna().sum().to_dict(),
    'sla_frescura_min': 30
}

print('Técnicos:', metadatos_tecnicos)
print('\nNegocio:', metadatos_negocio)
print('\nOperacionales:', metadatos_operacionales)

## 4. Data Lineage (Linaje de Datos)

**Linaje:** Visualización del flujo de datos desde origen hasta destino, incluyendo transformaciones.

### Niveles:
- **Table-level:** Tabla A → Transformación → Tabla B.
- **Column-level:** `clientes.email` → `hash(email)` → `anonimizado.email_hash`.

### Usos:
- **Auditoría:** Rastrear origen de un dato.
- **Impacto:** Si cambio tabla X, ¿qué reportes/tablas se afectan?
- **Cumplimiento:** Demostrar trazabilidad de PII (GDPR).

## 5. Ejemplo de Linaje (Conceptual)

In [ ]:
# Representación simple de linaje (grafo)
lineage_graph = {
    'bronze.ventas_raw': {
        'destinos': ['silver.ventas_limpias'],
        'transformacion': 'Filtrar, deduplicar, validar'
    },
    'silver.ventas_limpias': {
        'destinos': ['gold.kpi_ventas_diarias', 'gold.reportes_region'],
        'transformacion': 'Agregación por día y región'
    },
    'gold.kpi_ventas_diarias': {
        'destinos': ['BI Dashboard'],
        'transformacion': 'Join con clientes, cálculo métricas'
    }
}

# Imprimir linaje
for tabla, info in lineage_graph.items():
    print(f"{tabla} →\n  Transformación: {info['transformacion']}")
    print(f"  Destinos: {', '.join(info['destinos'])}\n")

## 6. Herramientas de Lineage

- **OpenLineage:** Estándar abierto para eventos de linaje (ya visto en Senior 03).
- **Apache Atlas:** Gobernanza con linaje integrado.
- **Amundsen / DataHub:** Catálogo con linaje automático.
- **dbt:** Lineage de transformaciones SQL (dbt docs).
- **Marquez:** Backend para OpenLineage.

## 7. Integración Catálogo + Linaje + Calidad

In [ ]:
# Simulación de entrada de catálogo enriquecida
entrada_catalogo = {
    'nombre': 'silver.ventas_limpias',
    'descripcion': 'Ventas validadas y deduplicadas',
    'owner': 'Equipo Datos',
    'tags': ['ventas', 'silver', 'confiable'],
    'esquema': [
        {'columna': 'fecha', 'tipo': 'date', 'descripcion': 'Fecha del pedido'},
        {'columna': 'pedido_id', 'tipo': 'string', 'descripcion': 'ID único'},
        {'columna': 'monto', 'tipo': 'float', 'descripcion': 'Monto total'}
    ],
    'lineage': {
        'upstream': ['bronze.ventas_raw'],
        'downstream': ['gold.kpi_ventas_diarias']
    },
    'calidad': {
        'completitud_pct': 99.5,
        'duplicados': 0,
        'ultima_validacion': '2025-11-07T06:00:00Z'
    },
    'acceso': {
        'clasificacion': 'interno',
        'roles_lectura': ['analista', 'bi']
    }
}

import json
print(json.dumps(entrada_catalogo, indent=2, ensure_ascii=False))

## 8. Ejercicio

Para un dataset de tu elección:
1. Define metadatos técnicos, de negocio y operacionales.
2. Dibuja (conceptual) el linaje: origen → transformación → destino.
3. Identifica 2 métricas de calidad relevantes y cómo las medirías.

In [ ]:
# Plantilla
mi_metadata = {
    'tecnicos': {},
    'negocio': {},
    'operacionales': {},
    'lineage': {'upstream': [], 'downstream': []},
    'calidad': {}
}
print('Completa el ejercicio aquí')

## 9. Resumen

- Metadatos son clave para gobernanza: técnicos, negocio, operacionales.
- Catálogo centraliza descubrimiento, documentación y políticas.
- Linaje (lineage) proporciona auditoría, impacto y cumplimiento.
- Integrar catálogo + linaje + calidad en una vista unificada mejora confianza y transparencia.